# EM Algorithm — Maximum Likelihood from Incomplete Data via the EM Algorithm (1977)

_Papers · Classical ML_

**A general two-step recipe — guess the hidden parts, then re-fit — that never lowers the likelihood and is still how we train mixture models.**

---

This notebook is a practice scaffold for the **EM Algorithm — Maximum Likelihood from Incomplete Data via the EM Algorithm (1977)** lesson. We build it up one step at a time — write the E-step and M-step from scratch, check them against a hand-worked example, fit a toy two-Gaussian mixture, then confirm the answer matches scikit-learn and that the log-likelihood only ever climbs. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Write the EM loop from scratch

EM alternates two steps until the fit stops improving. The **E-step** computes each point's *responsibility* `gamma[i,k]` — the posterior probability it came from cluster `k`. The **M-step** then re-fits each Gaussian's weight, mean, and variance using those soft assignments. We also record the observed-data log-likelihood at the start of every iteration, since the paper's Theorem 1 guarantees it never decreases.

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

def gauss(x, m, v):                     # 1-D Gaussian density N(x; mean m, variance v)
    return np.exp(-0.5 * (x - m)**2 / v) / np.sqrt(2 * np.pi * v)

def em_gmm(x, K, mu0, var0, pi0, iters=40):
    """EM for a 1-D Gaussian Mixture Model, from scratch. Returns (params, loglik history)."""
    pi = np.array(pi0, float)
    mu = np.array(mu0, float)
    var = np.array(var0, float)
    lls = []
    for _ in range(iters):
        # ---- E-step: responsibilities gamma[i,k] = pi_k N(x_i;mu_k,var_k) / sum_j (...) ----
        comp = np.stack([pi[k] * gauss(x, mu[k], var[k]) for k in range(K)], axis=1)  # (N,K)
        tot = comp.sum(axis=1)                          # mixture density per point
        lls.append(np.log(tot).sum())                   # observed-data log-likelihood (before M-step)
        gamma = comp / tot[:, None]                     # normalize each row -> sums to 1
        # ---- M-step: weighted updates ----
        Nk = gamma.sum(axis=0)                           # effective (soft) count per cluster
        pi = Nk / len(x)
        mu = (gamma * x[:, None]).sum(axis=0) / Nk
        var = (gamma * (x[:, None] - mu)**2).sum(axis=0) / Nk
    return (pi, mu, var), np.array(lls)

### Step 2 — Check one E-step and M-step on a worked example

Before trusting the loop, we reproduce the lesson's hand-worked numbers on three points. One E-step gives the responsibility of each point to cluster 0; one M-step mean update then re-centers cluster 0 from those soft counts. Matching the printed values confirms our E-step and M-step formulas are correct.

In [ ]:
# Recompute the WORKED EXAMPLE: one E-step responsibility + one M-step mean update.
xw = np.array([0.5, 1.5, 5.0])
pi_w = [0.5, 0.5]
mu_w = [0.0, 4.0]
var_w = [1.0, 1.0]

comp = np.stack([pi_w[k] * gauss(xw, mu_w[k], var_w[k]) for k in range(2)], axis=1)
gw = comp / comp.sum(1, keepdims=True)
print("worked gamma[:,0] :", [round(v, 6) for v in gw[:, 0]])   # [0.997527, 0.880797, 6e-06]

N0 = gw[:, 0].sum()
mu0_new = (gw[:, 0] * xw).sum() / N0
print("worked N0, mu0_new:", round(N0, 6), round(mu0_new, 6))   # 1.878331  0.96894

### Step 3 — Fit a toy two-Gaussian mixture and verify monotonic likelihood

We generate 300 points from two 1-D Gaussians and run our EM from a deliberately mediocre initialization. We then check that the log-likelihood rose at every iteration (Theorem 1) and recover the means, variances, and mixing weights — sorted by mean so they are easy to read.

In [ ]:
# Toy 1-D two-Gaussian data.
rng = np.random.default_rng(0)
n = 300
z = rng.random(n) < 0.4
x = np.where(z, rng.normal(-2.0, 0.8, n), rng.normal(3.0, 1.2, n))

# Run our EM from a deliberately mediocre init.
(pi, mu, var), lls = em_gmm(x, K=2, mu0=[-1.0, 1.0], var0=[1.0, 1.0], pi0=[0.5, 0.5])

print("monotonic LL increase:", bool(np.all(np.diff(lls) >= -1e-9)))   # True (Theorem 1)
print("LL[:5]:", [round(v, 2) for v in lls[:5]])  # [-1058.88, -612.49, -606.8, -606.74, -606.73]

o = np.argsort(mu)
print("scratch  LL=%.4f  mu=%s  var=%s  pi=%s"
      % (lls[-1], np.round(mu[o], 4), np.round(var[o], 4), np.round(pi[o], 4)))

### Step 4 — Confirm against the scikit-learn oracle

Our from-scratch fit should match a trusted reference. We fit `sklearn.mixture.GaussianMixture` on the same data and compare the converged log-likelihood. Agreement to within `1e-2` is our oracle check — proof the scratch implementation is correct, not just plausible.

In [ ]:
# THE ORACLE: must match sklearn.mixture.GaussianMixture on the same data.
gm = GaussianMixture(n_components=2, covariance_type='full', tol=1e-6,
                     reg_covar=1e-9, max_iter=200, random_state=0).fit(x.reshape(-1, 1))
sk_ll = gm.score(x.reshape(-1, 1)) * n

os = np.argsort(gm.means_.ravel())
print("sklearn  LL=%.4f  mu=%s  var=%s  pi=%s"
      % (sk_ll, np.round(gm.means_.ravel()[os], 4),
         np.round(gm.covariances_.ravel()[os], 4), np.round(gm.weights_[os], 4)))
print("LL matches sklearn (<1e-2):", abs(lls[-1] - sk_ll) < 1e-2)          # True
# scratch  LL=-606.7329  mu=[-2.0209  2.9588] ...   sklearn LL=-606.7329  mu=[-2.0207  2.9589] ...

## Visualize the data & results

_Run our from-scratch EM on a toy 1-D two-Gaussian mixture and record the observed-data log-likelihood at every iteration — does it rise monotonically and never dip, exactly as the paper's Theorem 1 (Eq. 3.7) promises?_

### Step 1 — Generate the data and run EM, logging the likelihood

Here we re-run a self-contained EM for 12 iterations on the same toy mixture, printing the observed-data log-likelihood at each step. The E-step and M-step are written out inline so you can see exactly what happens each pass. Watch the numbers: they should climb every iteration and flatten as EM converges.

In [ ]:
import numpy as np

def gauss(x, m, v):
    return np.exp(-0.5 * (x - m)**2 / v) / np.sqrt(2 * np.pi * v)

rng = np.random.default_rng(0)
n = 300
z = rng.random(n) < 0.4
x = np.where(z, rng.normal(-2.0, 0.8, n), rng.normal(3.0, 1.2, n))

pi = np.array([0.5, 0.5])
mu = np.array([-1.0, 1.0])
var = np.array([1.0, 1.0])
lls = []

for it in range(12):
    comp = np.stack([pi[k] * gauss(x, mu[k], var[k]) for k in range(2)], axis=1)  # E-step
    tot = comp.sum(axis=1)
    lls.append(np.log(tot).sum())                                                 # record LL
    gamma = comp / tot[:, None]                                                   # responsibilities
    Nk = gamma.sum(axis=0)                                                         # M-step
    pi = Nk / n
    mu = (gamma * x[:, None]).sum(0) / Nk
    var = (gamma * (x[:, None] - mu)**2).sum(0) / Nk
    print(f"iter {it:2d}  log-likelihood {lls[-1]:.2f}")

print("monotonic:", bool(np.all(np.diff(lls) >= -1e-9)))   # True — Theorem 1

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** One E-step responsibility. $K=2$, $\pi=(0.5,0.5)$, means $(0,4)$, variances $(1,1)$. For the point $x=1.5$, compute $\gamma$ to cluster 0. (Use $\mathcal{N}(1.5;0,1)=0.129518$, $\mathcal{N}(1.5;4,1)=0.017528$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Weight by $\pi$: cluster-0 term $=0.5\cdot0.129518=0.064759$; cluster-1 term $=0.5\cdot0.017528=0.008764$. — _Numerator of each responsibility = mixing weight times that cluster's density._
- Denominator $=0.064759+0.008764=0.073523$. — _Sum over clusters so the row sums to 1._
- $\gamma_{0}=0.064759/0.073523=0.880797$. — _Posterior probability point came from cluster 0._

**Answer:** $\gamma\approx(0.8808,\,0.1192)$. The point at $1.5$ is closer to the cluster-0 mean (0) than to cluster-1's (4), so cluster 0 takes ~88% responsibility — matching the worked example's middle row.

</details>

**Problem 2.** M-step mean for cluster 1. Using the three-point worked example, the cluster-1 responsibilities are $\gamma_{\cdot,1}=(0.002473,\,0.119203,\,0.999994)$ for $x=(0.5,1.5,5.0)$. Compute $N_1$ and $\mu_1^{\text{new}}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $N_1=0.002473+0.119203+0.999994=1.121670$. — _Effective (soft) count = total responsibility mass of cluster 1._
- Numerator $=0.002473\cdot0.5+0.119203\cdot1.5+0.999994\cdot5.0=0.001237+0.178805+4.99997=5.18001$. — _Responsibility-weighted sum of the points._
- $\mu_1^{\text{new}}=5.18001/1.121670=4.61813$. — _Weighted mean = numerator / effective count._

**Answer:** $N_1\approx1.1217$, $\mu_1^{\text{new}}\approx4.6181$. The point at $5.0$ dominates because it carries almost all of cluster 1's responsibility, pulling the new mean up toward it.

</details>

**Problem 3.** Ablation — 1 iteration vs convergence, and choosing $k$. In the CODE/CODEVIZ run, the observed-data log-likelihood goes from $-1058.88$ at init to $-612.49$ after one full EM step, then to $-606.73$ at convergence. Separately, sklearn's converged log-likelihood is $-708.32$ for $k=1$, $-606.73$ for $k=2$, $-604.78$ for $k=3$. What do these tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare 1 step ($-612.49$) to converged ($-606.73$). — _One iteration already does most of the work here, but is not fully converged — EM keeps climbing._
- Check the curve never dips between init and convergence. — _Theorem 1: monotone increase. $-1058.88\to-612.49\to-606.73$ is strictly rising._
- Compare $k=1,2,3$ log-likelihoods AND their BIC ($1428.0,\ 1242.0,\ 1255.2$). — _More components can only raise the (training) likelihood, so you cannot pick $k$ by likelihood alone — penalize complexity._

**Answer:** The log-likelihood rises every step (monotone, as Theorem 1 promises) and one step is a good-but-not-final approximation. Raw log-likelihood always improves with more components ($-708.32<-606.73<-604.78$), so it cannot choose $k$; a complexity-penalized score does — BIC is lowest at $k=2$ (1242.0), correctly recovering the two true clusters. (Our small run, not the paper's numbers.)

</details>